In [ ]:
import pandas as pd

In [2]:
def preprocess_flight_data(df):
    flight_cols = [
        "FL_DATE", "AIRLINE", "AIRLINE_CODE", "FL_NUMBER", "ORIGIN", "DEST",
        "CRS_DEP_TIME", "CRS_ARR_TIME", "CRS_ELAPSED_TIME", "DISTANCE",
        "CANCELLED", "DEP_DELAY", "AIRPORT_NAME"
    ]
    df = df[flight_cols]

    df = df[df["CANCELLED"] == 0]
    df = df.drop(columns=["CANCELLED"])
    
    # Chuyển đổi cột ngày tháng
    df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])
    df["day_of_week"] = df["FL_DATE"].dt.dayofweek
    df["month"] = df["FL_DATE"].dt.month

    season_map = {1: 0, 2: 0, 3: 1, 4: 1, 5: 1, 6: 2, 7: 2, 8: 2, 9: 3, 10: 3, 11: 3, 12: 0}
    df["season"] = df["month"].map(season_map)

    # Tạo nhãn mục tiêu: chuyến bay bị delay trên 15 phút
    df["delay_label"] = (df["DEP_DELAY"] > 30).astype(int)

    return df.drop(columns=["DEP_DELAY", "FL_NUMBER"])

In [3]:
def preprocess_weather_data(weather_df):
    """ Tiền xử lý dữ liệu thời tiết để chọn trạm có nhiều giá trị hợp lệ nhất """

    # Đếm số lượng giá trị khác 0 trên mỗi hàng
    weather_features = weather_df.columns[3:]
    weather_df["valid_count"] = (weather_df[weather_features] != 0).sum(axis=1)

    weather_df = weather_df.sort_values(by=["IATA_CODE", "date", "valid_count"], ascending=[True, True, False])
    weather_df = weather_df.drop_duplicates(subset=["IATA_CODE", "date"], keep="first")

    weather_df = weather_df.rename(columns={"date": "FL_DATE"})
    weather_df["FL_DATE"] = pd.to_datetime(weather_df["FL_DATE"])

    print(weather_df.head(20))
    return weather_df.drop(columns=["valid_count"])

In [4]:
def merge_data(flight_df, weather_df):
    weather_df = preprocess_weather_data(weather_df)

    merged_df = flight_df.merge(weather_df, left_on=["FL_DATE", "ORIGIN"], right_on=["FL_DATE", "IATA_CODE"], how="left")
    merged_df = merged_df.drop(columns=["IATA_CODE"])

    return merged_df

In [5]:
df_flight = pd.read_csv("filtered_flight.csv")
df_weather = pd.read_csv("no_na_weather.csv")

df_flight = preprocess_flight_data(df_flight)
df_merge = merge_data(df_flight, df_weather)
df_merge.dropna(inplace=True)

print(df_merge.shape)
print(df_merge.isna().any().any())

   IATA_CODE         station_id    FL_DATE   TAVG  TMAX  TMIN  AWND  WSF2  \
0        ABE  GHCND:USW00014737 2022-01-01  10.30  11.7   8.9   1.6   4.5   
1        ABE  GHCND:USW00014737 2022-01-02   6.70  13.9  -0.5   3.9   8.9   
2        ABE  GHCND:USW00014737 2022-01-03  -3.85   0.0  -7.7   5.0   9.4   
3        ABE  GHCND:USW00014737 2022-01-04  -3.85   1.1  -8.8   2.0   6.3   
4        ABE  GHCND:USW00014737 2022-01-05  -0.55   4.4  -5.5   1.1   4.0   
5        ABE  GHCND:USW00014737 2022-01-06  -1.60   1.7  -4.9   3.3   8.9   
6        ABE  GHCND:USW00014737 2022-01-07  -4.10  -1.6  -6.6   4.7   9.4   
7        ABE  GHCND:USW00014737 2022-01-08  -6.85  -2.7 -11.0   2.4   7.6   
8        ABE  GHCND:USW00014737 2022-01-09  -1.90   2.2  -6.0   3.6   6.3   
9        ABE  GHCND:USW00014737 2022-01-10  -2.20   2.2  -6.6   5.5   8.9   
10       ABE  GHCND:USW00014737 2022-01-11  -9.35  -6.0 -12.7   3.8   9.8   
11       ABE  GHCND:USW00014737 2022-01-12  -6.05   0.6 -12.7   1.8   5.4   

In [6]:
print(df_merge.head())

      FL_DATE                 AIRLINE AIRLINE_CODE ORIGIN DEST  CRS_DEP_TIME  \
1  2022-07-22   United Air Lines Inc.           UA    DEN  MSP           954   
8  2022-07-14        Republic Airline           YX    DCA  CHS          1346   
18 2022-06-10  Southwest Airlines Co.           WN    PHX  LAX          1350   
21 2022-08-15       PSA Airlines Inc.           OH    BUF  DCA          1731   
31 2022-08-08         JetBlue Airways           B6    DCA  SJU           815   

    CRS_ARR_TIME  CRS_ELAPSED_TIME  DISTANCE  \
1           1252             118.0     680.0   
8           1529             103.0     444.0   
18          1510              80.0     370.0   
21          1844              73.0     296.0   
31          1202             227.0    1554.0   

                                AIRPORT_NAME  ...  WT01  WT02  WT03  WT04  \
1               Denver International Airport  ...   0.0   0.0   0.0   0.0   
8       Ronald Reagan Washington Ntl Airport  ...   0.0   0.0   0.0   0.0   

In [6]:
print(df_merge.columns)

Index(['FL_DATE', 'AIRLINE', 'AIRLINE_CODE', 'ORIGIN', 'DEST', 'CRS_DEP_TIME',
       'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'DISTANCE', 'AIRPORT_NAME',
       'day_of_week', 'month', 'season', 'delay_label', 'station_id', 'TAVG',
       'TMAX', 'TMIN', 'AWND', 'WSF2', 'WSF5', 'WSFG', 'PRCP', 'SNOW', 'SNWD',
       'WT01', 'WT02', 'WT03', 'WT04', 'WT05', 'WT06', 'WT07', 'WT09', 'WT10',
       'WT11'],
      dtype='object')


In [7]:
df_merge["FL_DATE"] = df_merge["FL_DATE"].dt.date
print(df_merge.dtypes)

FL_DATE              object
AIRLINE              object
AIRLINE_CODE         object
ORIGIN               object
DEST                 object
CRS_DEP_TIME          int64
CRS_ARR_TIME          int64
CRS_ELAPSED_TIME    float64
DISTANCE            float64
AIRPORT_NAME         object
day_of_week           int32
month                 int32
season                int64
delay_label           int64
station_id           object
TAVG                float64
TMAX                float64
TMIN                float64
AWND                float64
WSF2                float64
WSF5                float64
WSFG                float64
PRCP                float64
SNOW                float64
SNWD                float64
WT01                float64
WT02                float64
WT03                float64
WT04                float64
WT05                float64
WT06                float64
WT07                float64
WT09                float64
WT10                float64
WT11                float64
dtype: object


In [7]:
import psycopg2
from dotenv import load_dotenv
import os
load_dotenv()

db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USERNAME")
db_password = os.getenv("DB_PWD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")

create_tables_query = """
CREATE TABLE IF NOT EXISTS dim_date (
    date DATE PRIMARY KEY,
    day_of_week INT NOT NULL,
    month INT NOT NULL,
    season INT NOT NULL
);

CREATE TABLE IF NOT EXISTS dim_airline (
    airline_id SERIAL PRIMARY KEY,
    airline_code VARCHAR(10) UNIQUE NOT NULL,
    airline_name VARCHAR(50) NOT NULL
);

CREATE TABLE IF NOT EXISTS dim_airport (
    airport_id SERIAL PRIMARY KEY,
    airport_code VARCHAR(10) UNIQUE NOT NULL,
    airport_name VARCHAR(100) NOT NULL
);

CREATE TABLE IF NOT EXISTS dim_weather (
    weather_id SERIAL PRIMARY KEY,
    station_id VARCHAR(25) NOT NULL,
    weather_date DATE NOT NULL,
    avg_temp FLOAT,
    max_temp FLOAT,
    min_temp FLOAT,
    wind_avg FLOAT,
    wind_max_2s FLOAT,
    wind_max_5s FLOAT,
    wind_max_gust FLOAT,
    precipitation FLOAT,
    snow FLOAT,
    snow_depth FLOAT,
    WT01 BOOLEAN,
    WT02 BOOLEAN,
    WT03 BOOLEAN,
    WT04 BOOLEAN,
    WT05 BOOLEAN,
    WT06 BOOLEAN,
    WT07 BOOLEAN,
    WT09 BOOLEAN,
    WT10 BOOLEAN,
    WT11 BOOLEAN,
    UNIQUE(station_id, weather_date)
);
CREATE TABLE IF NOT EXISTS fact_flights (
    flight_id SERIAL PRIMARY KEY,
    flight_date DATE NOT NULL REFERENCES dim_date(date),
    airline_id INT NOT NULL REFERENCES dim_airline(airline_id),
    origin_airport_id INT NOT NULL REFERENCES dim_airport(airport_id),
    dest_airport_id INT NOT NULL REFERENCES dim_airport(airport_id),
    crs_dep_time INT NOT NULL,
    crs_arr_time INT NOT NULL,
    crs_elapsed_time INT NOT NULL,
    distance FLOAT NOT NULL,
    weather_id INT REFERENCES dim_weather(weather_id),
    delay_label BOOLEAN NOT NULL,
    UNIQUE (flight_date, airline_id, origin_airport_id, dest_airport_id, crs_dep_time, delay_label)
);
"""


try:
    conn = psycopg2.connect(dbname=db_name, user=db_user, password=db_password, host=db_host, port=db_port)
    cur = conn.cursor()

    cur.execute(create_tables_query)
    print("Tables created successfully")
    conn.commit()

    # Insert into dim_date
    print("insert in to dim_date")
    for _, row in df_merge.iterrows():
        cur.execute("""
            INSERT INTO dim_date (date, day_of_week, month, season)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (date) DO NOTHING
        """, (row['FL_DATE'], row['day_of_week'], row['month'], row['season']))
    
    # Insert into dim_airline
    print("insert in to dim_airline")
    for _, row in df_merge.iterrows():
        cur.execute("""
            INSERT INTO dim_airline (airline_code, airline_name)
            VALUES (%s, %s)
            ON CONFLICT (airline_code) DO NOTHING
        """, (row['AIRLINE_CODE'], row['AIRLINE']))

    # Insert into dim_airport (ORIGIN and DEST)
    print("insert in to dim_airport")
    for _, row in df_merge.iterrows():
        cur.execute("""
            INSERT INTO dim_airport (airport_code, airport_name)
            VALUES (%s, %s)
            ON CONFLICT (airport_code) DO NOTHING
        """, (row['ORIGIN'], row['AIRPORT_NAME']))

        cur.execute("""
            INSERT INTO dim_airport (airport_code, airport_name)
            VALUES (%s, %s)
            ON CONFLICT (airport_code) DO NOTHING
        """, (row['DEST'], row['AIRPORT_NAME']))

    # Insert into dim_weather
    print("insert in to dim_weather")
    for _, row in df_merge.iterrows():
        cur.execute("""
            INSERT INTO dim_weather (
                station_id, weather_date, avg_temp, max_temp, min_temp, 
                wind_avg, wind_max_2s, wind_max_5s, wind_max_gust, 
                precipitation, snow, snow_depth, 
                WT01, WT02, WT03, WT04, WT05, WT06, WT07, WT09, WT10, WT11
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (station_id, weather_date) DO NOTHING
        """, (
            row['station_id'], row['FL_DATE'], row['TAVG'], row['TMAX'], row['TMIN'],
            row['AWND'], row['WSF2'], row['WSF5'], row['WSFG'],
            row['PRCP'], row['SNOW'], row['SNWD'],
            bool(row['WT01']), bool(row['WT02']), bool(row['WT03']), bool(row['WT04']),
            bool(row['WT05']), bool(row['WT06']), bool(row['WT07']), bool(row['WT09']),
            bool(row['WT10']), bool(row['WT11'])
        ))

    # Insert into fact_flights
    print("insert in to fact_flights")
    for _, row in df_merge.iterrows():
        # Get the airline_id
        cur.execute("SELECT airline_id FROM dim_airline WHERE airline_code = %s", (row['AIRLINE_CODE'],))
        airline_id = cur.fetchone()[0]

        # Get origin_airport_id
        cur.execute("SELECT airport_id FROM dim_airport WHERE airport_code = %s", (row['ORIGIN'],))
        origin_airport_id = cur.fetchone()[0]

        # Get dest_airport_id
        cur.execute("SELECT airport_id FROM dim_airport WHERE airport_code = %s", (row['DEST'],))
        dest_airport_id = cur.fetchone()[0]

        # Get weather_id
        cur.execute("SELECT weather_id FROM dim_weather WHERE station_id = %s AND weather_date = %s", (row['station_id'], row['FL_DATE']))
        weather_id = cur.fetchone()[0]

        # Insert into fact_flights
        cur.execute("""
        INSERT INTO fact_flights (flight_date, airline_id, origin_airport_id, dest_airport_id, crs_dep_time, crs_arr_time, crs_elapsed_time, distance, weather_id, delay_label)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (flight_date, airline_id, origin_airport_id, dest_airport_id, crs_dep_time, delay_label) DO NOTHING
    """, (row['FL_DATE'], airline_id, origin_airport_id, dest_airport_id, row['CRS_DEP_TIME'], row['CRS_ARR_TIME'], row['CRS_ELAPSED_TIME'], row['DISTANCE'], weather_id, bool(row['delay_label'])))

    # Commit the transaction
    print("Data inserted successfully")
    conn.commit()
except Exception as error:
    print(error)
finally:
    if cur is not None:
        cur.close()
    if conn is not None:
        conn.close()

Tables created successfully
insert in to dim_date


KeyboardInterrupt: 